# 프로젝트 경로와 재현 환경

현재 폴더 구조를 사용하고, 과거 입력 명세는 파일을 고치지 않고 메모리에서 경로를 해석합니다. 각 노트북이 `project-runtime` 셀만 불러옵니다. 원자료·모형·수치 해시는 변경하지 않습니다.

In [ ]:
from pathlib import Path as _ProjectPath
import json as _project_json
import hashlib as _project_hashlib
import os as _project_os

_project_start = _ProjectPath.cwd().resolve()
PROJECT_ROOT = next(p for p in [_project_start, *_project_start.parents]
                    if (p / 'environment/path_aliases.json').is_file())
_project_override = _project_os.environ.get('GRAD_PROJECT_ROOT')
if _project_override:
    PROJECT_ROOT = _ProjectPath(_project_override).resolve()
    if not (PROJECT_ROOT / 'environment/path_aliases.json').is_file():
        raise FileNotFoundError('GRAD_PROJECT_ROOT에 프로젝트 환경 파일이 없습니다.')
_project_aliases = _project_json.loads((PROJECT_ROOT / 'environment/path_aliases.json').read_text(encoding='utf-8'))
_project_migration_path = PROJECT_ROOT / 'environment/path_migration.json'
_project_changes = (_project_json.loads(_project_migration_path.read_text(encoding='utf-8')).get('changes', {})
                    if _project_migration_path.is_file() else {})
NODE_EXECUTABLE = str(PROJECT_ROOT / 'environment/node/node.exe')


## 현재 경로로 해석

상대 경로만 대응시킵니다. 현재 위치는 그대로 유지하며, 역사적 `reports/analysis_history/` 아래의 내부 구조는 바꾸지 않습니다.

In [ ]:
def project_relative(value):
    if not isinstance(value, str) or '\n' in value:
        return value
    original = value
    value = value.replace('\\', '/')
    if value in _project_aliases['exact']:
        return _project_aliases['exact'][value]
    for old, new in _project_aliases['prefixes']:
        if value.startswith(old + '/') or (value == old and '/' in old):
            return new + value[len(old):]
    if value in ('reports/20261005', 'reports/20261006', 'reports/kopis_collection', 'reports/seoul_collection'):
        return value
    if value.startswith('reports/') and not value.startswith((
            'reports/20261005/', 'reports/20261006/', 'reports/kopis_collection/', 'reports/seoul_collection/')):
        return 'reports/20261005/' + value[len('reports/'):]
    return original


def project_path(value):
    path = _ProjectPath(value)
    if path.is_absolute():
        return path
    return globals().get('ROOT', PROJECT_ROOT) / project_relative(str(value))


## 승인된 경로 수정의 해시 이력

과거 계획의 해시와 이관 전 해시가 같은 경우에만 변경 기록을 적용합니다. 실행 출력만 달라진 노트북은 승인된 코드·설명 해시도 확인합니다. 데이터나 통계 결과의 해시는 대체하지 않습니다.

In [ ]:
def project_notebook_source_hash(path):
    notebook = _project_json.loads(_ProjectPath(path).read_text(encoding='utf-8-sig'))
    source = [(cell['cell_type'], ''.join(cell.get('source', []))) for cell in notebook['cells']]
    return _project_hashlib.sha256(_project_json.dumps(source, ensure_ascii=False).encode('utf-8')).hexdigest()


def _project_migrated_digest(relative, digest):
    record = _project_changes.get(project_relative(relative))
    if not record or digest not in [record['before_sha256'], *record.get('accepted_before_sha256', [])]:
        return digest
    path = project_path(relative)
    actual = _project_hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != record['after_sha256']:
        if path.suffix != '.ipynb' or project_notebook_source_hash(path) != record.get('source_sha256'):
            raise ValueError(f'승인된 경로 수정 이후 파일이 변경되었습니다: {relative}')
    return actual


In [ ]:
def _project_resolve_metadata(value):
    if isinstance(value, list):
        return [_project_resolve_metadata(item) for item in value
                if not (isinstance(item, dict) and item.get('path') in _project_aliases['removed'])]
    if isinstance(value, dict):
        resolved = {}
        for key, item in value.items():
            # 사용자가 삭제한 수집 실행기만 과거 보호 목록에서 제외합니다.
            if key in _project_aliases['removed'] and isinstance(item, str) and len(item) == 64:
                continue
            new_key = project_relative(key)
            resolved[new_key] = (_project_migrated_digest(key, item)
                                if isinstance(item, str) and len(item) == 64 and '/' in key
                                else _project_resolve_metadata(item))
        if isinstance(value.get('path'), str):
            for key in ('sha256', 'sha256_before', 'sha256_after', 'expected_sha256'):
                if isinstance(value.get(key), str):
                    resolved[key] = _project_migrated_digest(value['path'], value[key])
        return resolved
    return project_relative(value) if isinstance(value, str) else value


def project_json_loads(payload, **kwargs):
    return _project_resolve_metadata(_project_json.loads(payload, **kwargs))


## 해석 확인

고정 입력 명세의 바이트와 해시는 유지됩니다. 해석된 경로는 현재 프로젝트를 기준으로 존재해야 합니다.

In [ ]:
examples = ['data/raw', 'final/notebooks', 'reports/final', 'test/notebooks']
[(old, project_relative(old), project_path(old).exists()) for old in examples]